In [2]:
pip install prophet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 17.5 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.3 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from prophet import Prophet
import numpy as np

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


Sap Flow Sensor 1

In [4]:
import pandas as pd
import numpy as np

# ==========================================
# STEP 1: Data Cleaning & Downsampling
# ==========================================
def process_step_1(file_path):
    print("Step 1: Loading raw data and downsampling to 15-minute intervals...")
    df = pd.read_csv(file_path)
    
    df['realdate'] = pd.to_datetime(df['realdate'])
    df = df.sort_values('realdate')
    
    # Average the 4 sensor points
    sensor_cols = ['svalue_1', 'svalue_2', 'svalue_3', 'svalue_4']
    df['sap_flow_mean'] = df[sensor_cols].mean(axis=1)
    
    # Downsample to 15-minute intervals
    df = df.set_index('realdate')
    df_15min = df[['sap_flow_mean']].resample('15min').mean()
    df_15min['sap_flow_mean'] = df_15min['sap_flow_mean'].interpolate(method='time')
    
    return df_15min.reset_index()

# ==========================================
# STEP 2: Backcasting (Average Daily Profile Method)
# ==========================================
def process_step_2(df_15min, start_date_july='2025-07-01'):
    print("Step 2: Calculating Average Daily Profile and backcasting July data...")
    
    df_15min = df_15min.copy()
    df_15min['time_of_day'] = df_15min['realdate'].dt.time
    
    # Calculate the exact 24-hour profile from your real sensors
    daily_profile = df_15min.groupby('time_of_day')['sap_flow_mean'].mean().reset_index()
    
    # Create blank dates for July
    actual_start = df_15min['realdate'].min()
    july_dates = pd.date_range(start=start_date_july, end=actual_start, freq='15min', inclusive='left')
    july_df = pd.DataFrame({'realdate': july_dates})
    
    # Map the profile onto July
    july_df['time_of_day'] = july_df['realdate'].dt.time
    july_df = pd.merge(july_df, daily_profile, on='time_of_day', how='left')
    
    # Cleanup and combine
    july_df = july_df.drop(columns=['time_of_day'])
    df_15min = df_15min.drop(columns=['time_of_day'])
    
    combined_df = pd.concat([july_df, df_15min], ignore_index=True)
    combined_df = combined_df.sort_values('realdate').reset_index(drop=True)
    
    return combined_df

# ==========================================
# STEP 3: The 7-Day Calibration Baseline
# ==========================================
def process_step_3(df_combined):
    print("Step 3: Establishing the 7-Day Calibration Baseline...")
    
    start_time = df_combined['realdate'].min()
    end_time = start_time + pd.Timedelta(days=7)
    
    calibration_data = df_combined[(df_combined['realdate'] >= start_time) & 
                                   (df_combined['realdate'] < end_time)].copy()
    
    calibration_data['time_of_day'] = calibration_data['realdate'].dt.time
    
    baseline = calibration_data.groupby('time_of_day')['sap_flow_mean'].quantile(0.25).reset_index()
    baseline = baseline.rename(columns={'sap_flow_mean': 'baseline_25th_pct'})
    
    return baseline

# ==========================================
# STEP 4: Trigger Logic & 10-Digit Rounding
# ==========================================
def process_step_4(df_combined, baseline):
    print("Step 4: Applying Trigger Logic and enforcing 10-digit precision...")
    
    df_combined = df_combined.sort_values('realdate').reset_index(drop=True)
    
    # 4-hour trailing average
    df_combined['4hr_trailing_avg'] = df_combined['sap_flow_mean'].rolling(window=16, min_periods=1).mean()
    
    df_combined['time_of_day'] = df_combined['realdate'].dt.time
    df_combined = pd.merge(df_combined, baseline, on='time_of_day', how='left')
    
    # THE ALERT: 4hr trailing avg strictly below the 25th percentile
    df_combined['is_stressed'] = df_combined['4hr_trailing_avg'] < df_combined['baseline_25th_pct']
    df_combined = df_combined.drop(columns=['time_of_day'])
    
    # ====== NEW: Round all numeric columns to 10 decimal places ======
    cols_to_round = ['sap_flow_mean', '4hr_trailing_avg', 'baseline_25th_pct']
    df_combined[cols_to_round] = df_combined[cols_to_round].round(10)
    
    return df_combined

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    file_path = "../data/sap/raw/sensor1.csv"  # raw input
    
    df_step1 = process_step_1(file_path)
    df_combined = process_step_2(df_step1, start_date_july='2025-07-01')
    baseline_df = process_step_3(df_combined)
    final_dataset = process_step_4(df_combined, baseline_df)
    
    # Save to processed location
    final_dataset.to_csv("../data/sap/sap_flow_sensor1.csv", index=False)
    
    print("\nSUCCESS! Pipeline complete.")
    print("Data saved to '../data/sap/sap_flow_sensor1.csv'\n")
    print(final_dataset.head())

Step 1: Loading raw data and downsampling to 15-minute intervals...
Step 2: Calculating Average Daily Profile and backcasting July data...
Step 3: Establishing the 7-Day Calibration Baseline...
Step 4: Applying Trigger Logic and enforcing 10-digit precision...

SUCCESS! Pipeline complete.
Data saved to 'sap_flow_sensor1.csv'

             realdate  sap_flow_mean  4hr_trailing_avg  baseline_25th_pct  \
0 2025-07-01 00:00:00       0.109174          0.109174           0.109174   
1 2025-07-01 00:15:00       0.108878          0.109026           0.108878   
2 2025-07-01 00:30:00       0.108007          0.108686           0.108007   
3 2025-07-01 00:45:00       0.106890          0.108237           0.106890   
4 2025-07-01 01:00:00       0.106001          0.107790           0.106001   

   is_stressed  
0        False  
1        False  
2        False  
3        False  
4        False  


/var/folders/6c/_hkw_cbx7j33k946pnhyt8yw0000gn/T/ipykernel_11887/328894557.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['realdate'] = pd.to_datetime(df['realdate'])


Sap Flow Sensor 2

In [5]:
import pandas as pd
import numpy as np

# ==========================================
# STEP 1: Data Cleaning & Downsampling
# ==========================================
def process_step_1(file_path):
    print("Step 1: Loading raw data and downsampling to 15-minute intervals...")
    df = pd.read_csv(file_path)
    
    df['realdate'] = pd.to_datetime(df['realdate'])
    df = df.sort_values('realdate')
    
    # Average the 4 sensor points
    sensor_cols = ['svalue_1', 'svalue_2', 'svalue_3', 'svalue_4']
    df['sap_flow_mean'] = df[sensor_cols].mean(axis=1)
    
    # Downsample to 15-minute intervals
    df = df.set_index('realdate')
    df_15min = df[['sap_flow_mean']].resample('15min').mean()
    df_15min['sap_flow_mean'] = df_15min['sap_flow_mean'].interpolate(method='time')
    
    return df_15min.reset_index()

# ==========================================
# STEP 2: Backcasting (Average Daily Profile Method)
# ==========================================
def process_step_2(df_15min, start_date_july='2025-07-01'):
    print("Step 2: Calculating Average Daily Profile and backcasting July data...")
    
    df_15min = df_15min.copy()
    df_15min['time_of_day'] = df_15min['realdate'].dt.time
    
    # Calculate the exact 24-hour profile from your real sensors
    daily_profile = df_15min.groupby('time_of_day')['sap_flow_mean'].mean().reset_index()
    
    # Create blank dates for July
    actual_start = df_15min['realdate'].min()
    july_dates = pd.date_range(start=start_date_july, end=actual_start, freq='15min', inclusive='left')
    july_df = pd.DataFrame({'realdate': july_dates})
    
    # Map the profile onto July
    july_df['time_of_day'] = july_df['realdate'].dt.time
    july_df = pd.merge(july_df, daily_profile, on='time_of_day', how='left')
    
    # Cleanup and combine
    july_df = july_df.drop(columns=['time_of_day'])
    df_15min = df_15min.drop(columns=['time_of_day'])
    
    combined_df = pd.concat([july_df, df_15min], ignore_index=True)
    combined_df = combined_df.sort_values('realdate').reset_index(drop=True)
    
    return combined_df

# ==========================================
# STEP 3: The 7-Day Calibration Baseline
# ==========================================
def process_step_3(df_combined):
    print("Step 3: Establishing the 7-Day Calibration Baseline...")
    
    start_time = df_combined['realdate'].min()
    end_time = start_time + pd.Timedelta(days=7)
    
    calibration_data = df_combined[(df_combined['realdate'] >= start_time) & 
                                   (df_combined['realdate'] < end_time)].copy()
    
    calibration_data['time_of_day'] = calibration_data['realdate'].dt.time
    
    baseline = calibration_data.groupby('time_of_day')['sap_flow_mean'].quantile(0.25).reset_index()
    baseline = baseline.rename(columns={'sap_flow_mean': 'baseline_25th_pct'})
    
    return baseline

# ==========================================
# STEP 4: Trigger Logic & 10-Digit Rounding
# ==========================================
def process_step_4(df_combined, baseline):
    print("Step 4: Applying Trigger Logic and enforcing 10-digit precision...")
    
    df_combined = df_combined.sort_values('realdate').reset_index(drop=True)
    
    # 4-hour trailing average
    df_combined['4hr_trailing_avg'] = df_combined['sap_flow_mean'].rolling(window=16, min_periods=1).mean()
    
    df_combined['time_of_day'] = df_combined['realdate'].dt.time
    df_combined = pd.merge(df_combined, baseline, on='time_of_day', how='left')
    
    # THE ALERT: 4hr trailing avg strictly below the 25th percentile
    df_combined['is_stressed'] = df_combined['4hr_trailing_avg'] < df_combined['baseline_25th_pct']
    df_combined = df_combined.drop(columns=['time_of_day'])
    
    # ====== NEW: Round all numeric columns to 10 decimal places ======
    cols_to_round = ['sap_flow_mean', '4hr_trailing_avg', 'baseline_25th_pct']
    df_combined[cols_to_round] = df_combined[cols_to_round].round(10)
    
    return df_combined

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    file_path = "../data/sap/raw/sensor2.csv"  # raw input
    
    df_step1 = process_step_1(file_path)
    df_combined = process_step_2(df_step1, start_date_july='2025-07-01')
    baseline_df = process_step_3(df_combined)
    final_dataset = process_step_4(df_combined, baseline_df)
    
    # Save to processed location
    final_dataset.to_csv("../data/sap/sap_flow_sensor2.csv", index=False)
    
    print("\nSUCCESS! Pipeline complete.")
    print("Data saved to '../data/sap/sap_flow_sensor2.csv'\n")
    print(final_dataset.head())

Step 1: Loading raw data and downsampling to 15-minute intervals...
Step 2: Calculating Average Daily Profile and backcasting July data...
Step 3: Establishing the 7-Day Calibration Baseline...
Step 4: Applying Trigger Logic and enforcing 10-digit precision...

SUCCESS! Pipeline complete.
Data saved to 'sap_flow_sensor2.csv'

             realdate  sap_flow_mean  4hr_trailing_avg  baseline_25th_pct  \
0 2025-07-01 00:00:00       0.056246          0.056246           0.056246   
1 2025-07-01 00:15:00       0.055653          0.055950           0.055653   
2 2025-07-01 00:30:00       0.055138          0.055679           0.055138   
3 2025-07-01 00:45:00       0.054415          0.055363           0.054415   
4 2025-07-01 01:00:00       0.053549          0.055000           0.053549   

   is_stressed  
0        False  
1        False  
2        False  
3        False  
4        False  


/var/folders/6c/_hkw_cbx7j33k946pnhyt8yw0000gn/T/ipykernel_11887/1122715207.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['realdate'] = pd.to_datetime(df['realdate'])
